In [1]:
!pip install yfinance pandas numpy tqdm

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
tickers = [
    # Historically distressed / high risk
    "AMC", "BYND", "PTON", "SEAT", "CMLS",
    "NKLA", "WKHS", "SPCE", "BBBY", "PLUG",
    # Stable / large cap (negative examples — equally important)
    "AAPL", "MSFT", "JNJ", "WMT",
    "KO", "PG", "HD", "V", "MCD"
]
# We fix the observation date at March 18, 2025 (our "t")
# Everything before this date goes into the features
# Everything after this date is what we're trying to predict
# This way we're not accidentally using future information — no look-ahead bias
# (following the forward-looking framework from Campbell et al. 2008)
OBSERVATION_DATE = "2025-03-18"  # our t

In [4]:
all_prices = []

for ticker in tickers:

    data = yf.download(ticker, start="2014-01-01", auto_adjust=True, progress=False)

    data.columns = data.columns.get_level_values(0)
    data = data.reset_index()

    data["ticker"] = ticker

    all_prices.append(data)

final_prices = pd.concat(all_prices)

In [ ]:
final_prices.head()

Price,Date,Close,High,Low,Open,Volume,ticker
0,2014-01-02,145.140244,150.606371,143.557944,148.304843,42580,AMC
1,2014-01-03,143.989471,145.859469,142.263328,144.564856,29870,AMC
2,2014-01-06,143.917557,144.564858,142.479100,144.205244,50970,AMC
3,2014-01-07,144.205261,145.068333,143.342190,144.133334,85680,AMC
4,2014-01-08,143.557892,144.564807,143.054424,144.564807,49790,AMC


In [ ]:
final_prices.shape

(52640, 7)

In [ ]:
final_prices.columns

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker'], dtype='object', name='Price')

In [ ]:
final_prices["ticker"].unique()

array(['AMC', 'BYND', 'PTON', 'SEAT', 'CMLS', 'NKLA', 'WKHS', 'SPCE',
       'BBBY', 'PLUG', 'AAPL', 'MSFT', 'JPM', 'JNJ', 'WMT', 'KO', 'PG',
       'HD', 'V', 'MCD'], dtype=object)

In [ ]:
final_prices.columns.name = None
final_prices.head()

,Date,Close,High,Low,Open,Volume,ticker
0,2014-01-02,145.140244,150.606371,143.557944,148.304843,42580,AMC
1,2014-01-03,143.989471,145.859469,142.263328,144.564856,29870,AMC
2,2014-01-06,143.917557,144.564858,142.479100,144.205244,50970,AMC
3,2014-01-07,144.205261,145.068333,143.342190,144.133334,85680,AMC
4,2014-01-08,143.557892,144.564807,143.054424,144.564807,49790,AMC


In [ ]:
final_prices = final_prices.sort_values(["ticker","Date"])
final_prices = final_prices.reset_index(drop=True)

In [ ]:
final_prices["return_12m"] = final_prices.groupby("ticker")["Close"].pct_change(252)

In [ ]:
final_prices["return_6m"] = final_prices.groupby("ticker")["Close"].pct_change(126)

In [ ]:
final_prices["daily_return"] = final_prices.groupby("ticker")["Close"].pct_change()

In [ ]:
final_prices["volatility_6m"] = (
    final_prices.groupby("ticker")["daily_return"]
    .rolling(126)
    .std()
    .reset_index(level=0, drop=True)
)
# volume_change_1m — 1-month change in trading volume
# abnormal volume shifts often precede distress events as investors react
final_prices["volume_change_1m"] = final_prices.groupby("ticker")["Volume"].pct_change(20)

In [ ]:
final_prices.head()

,Date,Close,High,Low,Open,Volume,ticker,return_12m,return_6m,daily_return,volatility_6m,volume_change_1m
0,2014-01-02,17.140659,17.261515,17.106263,17.219681,234684800,AAPL,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,16.764151,17.158323,16.747106,17.132294,392467600,AAPL,NaN,NaN,-0.021966,NaN,NaN
2,2014-01-06,16.855564,16.944500,16.535453,16.654759,412610800,AAPL,NaN,NaN,0.005453,NaN,NaN
3,2014-01-07,16.735023,16.918475,16.669328,16.867654,317209200,AAPL,NaN,NaN,-0.007151,NaN,NaN
4,2014-01-08,16.841005,16.906080,16.693191,16.696908,258529600,AAPL,NaN,NaN,0.006333,NaN,NaN


In [ ]:
final_prices = final_prices.dropna()

## NaN Handling Strategy

When we compute rolling features, the early rows for each ticker
come out as NaN — this is expected because rolling windows need
enough historical data before they can produce a value.

For example:
- return_12m needs 252 trading days of history (1 full year)
- volatility_6m needs 126 trading days of history (6 months)

These early NaN rows don't carry any useful information, so we
just drop them.

**Decision: Drop all rows with NaN values using `dropna()`.**

Note: The balance sheet NaNs (interest_coverage and
net_income_to_total_assets) are a different case — these aren't
missing data, they just mean the company has little to no interest
expense (e.g. AAPL, HD, MCD are essentially debt-free). We handle
these separately in the ML pipeline using median imputation, so we
don't lose these companies from the dataset. Dropping them would
mean losing some of our cleanest stable examples.

In [ ]:
final_prices.shape

(47819, 12)

In [ ]:
# ── NEW: Pull balance sheet features from yfinance ────────────────────────────

def get_balance_sheet_features(ticker):
    try:
        stock = yf.Ticker(ticker)
        bs  = stock.quarterly_balance_sheet
        inc = stock.quarterly_income_stmt
        if bs.empty or inc.empty:
            return None

        total_liabilities = bs.loc['Total Liabilities Net Minority Interest'].iloc[0]
        total_assets      = bs.loc['Total Assets'].iloc[0]
        current_assets    = bs.loc['Current Assets'].iloc[0]
        current_liab      = bs.loc['Current Liabilities'].iloc[0]
        net_income        = inc.loc['Net Income'].iloc[0] if 'Net Income' in inc.index else None
        ebit              = inc.loc['EBIT'].iloc[0] if 'EBIT' in inc.index else None

        interest_exp = None
        for label in ['Interest Expense', 'Interest Expense Non Operating']:
            if label in inc.index:
                interest_exp = abs(inc.loc[label].iloc[0])
                break

        return {
            'ticker': ticker,
            'total_liabilities_to_total_assets': total_liabilities / total_assets if total_assets else None,
            'current_ratio':                     current_assets / current_liab if current_liab else None,
            'interest_coverage':                 ebit / interest_exp if (ebit and interest_exp and interest_exp != 0) else None,
            'net_income_to_total_assets':        net_income / total_assets if (net_income is not None and total_assets) else None,
        }
    except Exception as e:
        print(f"  ⚠ {ticker}: {e}")
        return None

bs_rows = []
for t in tickers:
    row = get_balance_sheet_features(t)
    if row:
        bs_rows.append(row)
        print(f"✅ {t}")
    else:
        print(f"❌ {t} — skipped")

bs_df = pd.DataFrame(bs_rows)
print(f"\nBalance sheet shape: {bs_df.shape}")
bs_df

✅ AMC
✅ BYND
✅ PTON
✅ SEAT
✅ CMLS
❌ NKLA — skipped
✅ WKHS
✅ SPCE
✅ BBBY
✅ PLUG
✅ AAPL
✅ MSFT
  ⚠ JPM: 'Current Assets'
❌ JPM — skipped
✅ JNJ
✅ WMT
✅ KO
✅ PG
✅ HD
✅ V
✅ MCD

Balance sheet shape: (18, 5)


,ticker,total_liabilities_to_total_assets,current_ratio,interest_coverage,net_income_to_total_assets
0,AMC,1.236324,0.412176,0.107595,-0.015890
1,BYND,2.307496,4.535338,-24.052286,-0.184582
2,PTON,1.150929,1.982926,-0.238095,-0.017925
3,SEAT,0.696952,0.671736,-3.736377,-0.007701
4,CMLS,1.052789,1.742267,-0.153186,-0.018927
5,WKHS,0.725274,1.207388,-45.619907,-0.067057
6,SPCE,0.735342,2.868429,-18.820615,-0.075470
7,BBBY,0.487767,1.249649,NaN,-0.049062
8,PLUG,0.613302,2.309688,-41.572498,-0.326054
9,AAPL,0.767491,0.973745,NaN,0.110987


In [ ]:
# ── IMPROVED: Multi-criteria distress label ───────────────────────────────────

def get_forward_return(ticker, obs_date, months=12):
    try:
        start = pd.Timestamp(obs_date)
        end   = start + pd.DateOffset(months=months) + pd.DateOffset(days=5)
        df_p  = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if df_p.empty:
            return None
        price = df_p['Close'].squeeze()
        return (price.iloc[-1] - price.iloc[0]) / price.iloc[0]
    except:
        return None

# ── Criterion 1: Price collapse ───────────────────────────────────────────────
snapshot = final_prices.groupby('ticker').last().reset_index()
snapshot = snapshot.merge(bs_df, on='ticker', how='inner')

snapshot['forward_return_12m'] = snapshot['ticker'].apply(
    lambda t: get_forward_return(t, OBSERVATION_DATE)
)
# We originally used -0.50 but relaxed it to -0.40 after checking the data
# PTON dropped -37% which felt distressed enough to include
# -0.40 still avoids flagging healthy companies like PG or WMT
price_distress = snapshot['forward_return_12m'] < -0.40

# ── Criterion 2: Known bankruptcy filings ─────────────────────────────────────
# We checked both of these on SEC EDGAR to confirm they actually filed
# BBBY filed April 2023 — the stock is basically worthless at this point
#   (filed before our observation date, so the price already reflects it)
# CMLS filed back in 2020, restructured, but still clearly struggling
bankrupt_tickers = [
    'BBBY',
    'CMLS',
]
bankruptcy_distress = snapshot['ticker'].isin(bankrupt_tickers)

# ── Criterion 3: Interest coverage < 1 ───────────────────────────────────────
# A ratio below 1 means EBIT < interest expense — the company literally
# cannot cover its debt payments from operations, which is a direct signal
# of financial distress (Altman 1968)
coverage_distress = snapshot['interest_coverage'] < 1

# ── Combine all three criteria (OR logic) ─────────────────────────────────────
snapshot['price_distress']    = price_distress.astype(int)
snapshot['bankrupt_distress'] = bankruptcy_distress.astype(int)
snapshot['coverage_distress'] = coverage_distress.astype(int)

snapshot['distress_label'] = (
    price_distress | bankruptcy_distress | coverage_distress
).astype(int)

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 55)
print("DISTRESS LABEL BREAKDOWN")
print("=" * 55)
print(snapshot[['ticker', 'forward_return_12m',
                'interest_coverage',
                'price_distress',
                'bankrupt_distress',
                'coverage_distress',
                'distress_label']].to_string(index=False))
print(f"\nTotal companies      : {len(snapshot)}")
print(f"Distressed (label=1) : {snapshot['distress_label'].sum()}")
print(f"Stable    (label=0)  : {(snapshot['distress_label']==0).sum()}")
print(f"Distress rate        : {snapshot['distress_label'].mean():.1%}")
print("\nBreakdown by criterion:")
print(f"  Price collapse           : {price_distress.sum()}")
print(f"  Bankruptcy filing        : {bankruptcy_distress.sum()}")
print(f"  Interest coverage < 1   : {coverage_distress.sum()}")

In [ ]:
# ── Save final dataset ────────────────────────────────────────────────────────

final_dataset = snapshot[[
    'ticker',
    'return_12m', 'return_6m', 'volatility_6m', 'volume_change_1m',
    'total_liabilities_to_total_assets', 'interest_coverage',
    'net_income_to_total_assets', 'current_ratio',
    'distress_label'
]].copy()

final_dataset.to_csv("corporate_distress_features_v2.csv", index=False)
print(f"✅ Saved: corporate_distress_features_v2.csv")
print(f"Shape: {final_dataset.shape}")
print(f"\nLabel distribution:")
print(final_dataset['distress_label'].value_counts())
print(f"\nDistressed companies:")
print(final_dataset[final_dataset['distress_label']==1]['ticker'].values)
print(f"\nStable companies:")
print(final_dataset[final_dataset['distress_label']==0]['ticker'].values)
final_dataset

✅ Saved: corporate_distress_features_v2.csv
Shape: (18, 10)

Label distribution:
distress_label
0    9
1    9
Name: count, dtype: int64

Distressed companies:
['AMC' 'BBBY' 'BYND' 'CMLS' 'PLUG' 'PTON' 'SEAT' 'SPCE' 'WKHS']

Stable companies:
['AAPL' 'HD' 'JNJ' 'KO' 'MCD' 'MSFT' 'PG' 'V' 'WMT']


,ticker,return_12m,return_6m,volatility_6m,volume_change_1m,total_liabilities_to_total_assets,interest_coverage,net_income_to_total_assets,current_ratio,distress_label
0,AAPL,0.173034,0.051507,0.014000,0.013135,0.767491,NaN,0.110987,0.973745,0
1,AMC,-0.661130,-0.648276,0.033843,-0.514830,1.236324,0.107595,-0.015890,0.412176,1
2,BBBY,-0.137809,-0.502548,0.044319,-0.193346,0.487767,NaN,-0.049062,1.249649,1
3,BYND,-0.801136,-0.741697,0.201101,0.481444,2.307496,-24.052286,-0.184582,4.535338,1
4,CMLS,-0.988891,-0.962963,0.149973,-0.587050,1.052789,-0.153186,-0.018927,1.742267,1
5,HD,-0.041892,-0.204420,0.014397,0.188569,0.885993,NaN,NaN,1.050863,0
6,JNJ,0.497734,0.360399,0.010064,-0.178493,0.590663,24.205607,0.025681,1.027676,0
7,KO,0.114835,0.162887,0.010510,-0.392187,0.672998,7.872390,0.021667,1.458766,0
8,MCD,0.062717,0.053114,0.010312,-0.411959,1.035688,NaN,NaN,1.000000,0
9,MSFT,0.015666,-0.227135,0.016279,0.110265,0.412485,66.551630,0.057805,1.386024,0


In [ ]:
from google.colab import files
files.download("corporate_distress_features_v2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>